# Phase 23 — Complementary performance metrics (balanced accuracy, sensitivity, specificity, PPV, NPV, PR-AUC)
**Resolves:** R2 minor-4 (accuracy is of limited value under class imbalance), supports R2-11 (per-outcome selected vs RF side by side).
For each dataset and for BOTH the proposed RF and the one-SE selected candidate: AUC, PR-AUC (with prevalence as the no-skill
baseline), accuracy, balanced accuracy, sensitivity, specificity, PPV, NPV, macro-F1, predicted-positive rate — all with
95% bootstrap CIs at the calibration-tuned (macro-F1) threshold.
**Outputs:** `S2_complementary_metrics_long.csv`, `S2_complementary_metrics_wide.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from modeling import tune_threshold
long_rows, wide_rows = [], []
KEYS = ["auc", "pr_auc", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "ppv", "npv", "macro_f1", "predicted_positive_rate"]
for tag in DATASETS:
    S = load_split(tag, ENV); y = S["pred_test"]["y_true"].values; yc = S["pred_cal"]["y_true"].values
    comp = pd.read_csv(os.path.join(ENV["MODELS"], "ensemble_final", tag, "strategy_comparison.csv")); sel, *_ = one_se_select(comp)
    for label, name in [("Proposed RF", "single_rf"), (f"1-SE selected ({sel})", sel)]:
        if label.startswith("1-SE") and name == "single_rf":
            continue                                    # identical to the proposed model
        p_cal = strategy_proba(name, S["oof"], S["pred_cal"], seed=SEED); p_te = strategy_proba(name, S["oof"], S["pred_test"], seed=SEED)
        thr, _ = tune_threshold(yc, p_cal)
        t = bootstrap_table(y, p_te, lambda a, b: operating_metrics(a, b, thr), B=B, seed=SEED)
        t.insert(0, "model", label); t.insert(0, "dataset", tag); t["threshold"] = thr; t["prevalence_no_skill_pr_auc"] = y.mean(); long_rows.append(t)
        row = dict(dataset=tag, model=label, threshold=round(thr, 2), n_test=len(y), prevalence=round(y.mean(), 3))
        for k in KEYS:
            r = t[t.metric == k].iloc[0]; row[k] = fmt_ci(r.estimate, r.ci_lower, r.ci_upper)
        wide_rows.append(row)
save_table(pd.concat(long_rows, ignore_index=True), ENV, "S2_complementary_metrics_long.csv")
wide = pd.DataFrame(wide_rows); save_table(wide, ENV, "S2_complementary_metrics_wide.csv"); wide

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_complementary_metrics_long.csv  (98 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_complementary_metrics_wide.csv  (7 rows)


,dataset,model,threshold,n_test,prevalence,auc,pr_auc,accuracy,balanced_accuracy,sensitivity,specificity,ppv,npv,macro_f1,predicted_positive_rate
0,pakistan,Proposed RF,0.51,2798,0.637,0.557 (0.535-0.579),0.679 (0.654-0.704),0.549 (0.530-0.567),0.539 (0.519-0.558),0.574 (0.552-0.598),0.503 (0.473-0.535),0.670 (0.646-0.693),0.402 (0.375-0.431),0.533 (0.514-0.551),0.546 (0.528-0.564)
1,pakistan,1-SE selected (single_svm),0.46,2798,0.637,0.562 (0.539-0.584),0.685 (0.660-0.709),0.556 (0.538-0.575),0.541 (0.521-0.560),0.598 (0.575-0.620),0.484 (0.453-0.516),0.670 (0.647-0.693),0.406 (0.378-0.435),0.537 (0.518-0.555),0.568 (0.549-0.586)
2,arab5_depression,Proposed RF,0.55,301,0.439,0.687 (0.625-0.747),0.558 (0.481-0.659),0.638 (0.581-0.694),0.631 (0.575-0.688),0.576 (0.491-0.660),0.686 (0.614-0.759),0.589 (0.500-0.676),0.674 (0.596-0.746),0.631 (0.575-0.688),0.429 (0.372-0.485)
3,arab5_depression,1-SE selected (single_knn),0.47,301,0.439,0.634 (0.568-0.697),0.508 (0.436-0.601),0.598 (0.542-0.654),0.612 (0.557-0.668),0.727 (0.651-0.803),0.497 (0.420-0.573),0.530 (0.455-0.607),0.700 (0.610-0.787),0.597 (0.540-0.654),0.601 (0.545-0.658)
4,arab5_anxiety,Proposed RF,0.45,191,0.723,0.782 (0.696-0.863),0.873 (0.804-0.933),0.796 (0.738-0.848),0.708 (0.638-0.778),0.906 (0.854-0.953),0.509 (0.380-0.638),0.828 (0.764-0.885),0.675 (0.515-0.822),0.723 (0.647-0.793),0.791 (0.733-0.848)
5,arab5_anxiety,1-SE selected (single_xgb),0.45,191,0.723,0.785 (0.702-0.863),0.878 (0.810-0.937),0.785 (0.723-0.843),0.706 (0.634-0.779),0.884 (0.830-0.936),0.528 (0.396-0.660),0.830 (0.764-0.890),0.636 (0.486-0.783),0.717 (0.642-0.788),0.770 (0.712-0.827)
6,uganda,Proposed RF,0.55,2841,0.481,0.697 (0.677-0.716),0.666 (0.637-0.694),0.636 (0.618-0.655),0.633 (0.615-0.651),0.541 (0.513-0.568),0.725 (0.701-0.749),0.645 (0.616-0.673),0.630 (0.608-0.654),0.631 (0.613-0.650),0.403 (0.385-0.420)
